# Chapter 8: Transactions
*From: Designing Data-Intensive Applications*

---

## TL;DR

- A **transaction** groups multiple reads and writes into a single logical unit that either fully commits or fully aborts, so applications don't have to reason about partial failures.
- The **ACID** acronym (atomicity, consistency, isolation, durability) describes the safety guarantees a transaction provides, but each letter is nuanced and "ACID-compliant" has become more marketing than specification.
- Isolation levels form a **ladder**: read uncommitted -> read committed -> snapshot isolation (a.k.a. repeatable read) -> serializable. Each level eliminates a class of concurrency anomaly at the cost of extra concurrency-control machinery.
- Beyond classic dirty reads lies a menagerie of harder races: **lost updates**, **write skew**, and **phantoms** — some only preventable with true serializability.
- True serializable isolation has three practical implementations: **literal serial execution** (VoltDB, single-threaded Redis), **two-phase locking (2PL)**, and **serializable snapshot isolation (SSI)** (PostgreSQL, CockroachDB).
- Across nodes, atomic commit requires a **distributed transaction** protocol like **two-phase commit (2PC)**. It works but blocks on coordinator failure; heterogeneous XA is frequently problematic, while database-internal 2PC (Spanner, CockroachDB) remains viable.

---

## Why Transactions Matter

Real data systems have to tolerate a long list of failure modes, any of which can corrupt state if the application tries to handle them by hand:

- The database software or hardware may crash mid-write.
- The application may crash halfway through a series of operations.
- The network may cut off the application from the database or nodes from each other.
- Multiple clients may write concurrently and overwrite each other's changes.
- A client may read data that has only partially been updated.
- Subtle race conditions between clients can produce surprising bugs.

Transactions exist to **simplify the programming model**: if something goes wrong, the transaction is aborted, the database guarantees nothing half-done sticks, and the application can safely retry. Abandoning transactions (as first-generation NoSQL did) pushes all of that complexity up into application code.

---

## The ACID Guarantees

| Letter | One-line definition | What it actually guarantees | Common misunderstanding |
|---|---|---|---|
| **Atomicity** | All-or-nothing on fault. | If a fault occurs mid-transaction, all writes are discarded, so retry is safe. "Abortability" would be a clearer name. | Not about concurrency — that's isolation. Not the same as "atomic operation" in concurrent programming. |
| **Consistency** | The database moves from one valid state to another. | Application-level invariants (e.g. debits = credits) hold **if the app writes correct transactions**. | It's really the application's job. Joe Hellerstein said the C "was tossed in to make the acronym work." |
| **Isolation** | Concurrent transactions don't step on each other. | Varies wildly by level — from "no dirty reads" to full serializability. | Most databases don't default to serializable; "isolation" alone tells you very little. |
| **Durability** | Once committed, it survives crashes. | Data is fsync'd to disk and/or replicated before commit returns. | On a single disk, "durable" still fails if the disk dies. Real durability needs replication + backups. |

Single-object operations (one row/document) are usually atomic and isolated by default. **Multi-object** atomicity and isolation require an explicit transaction — this is what many NoSQL stores dropped.

---

## Isolation Levels Hierarchy

```mermaid
graph LR
  RU[Read Uncommitted] --> RC[Read Committed]
  RC --> SI[Snapshot Isolation / Repeatable Read]
  SI --> SER[Serializable]
  style RU fill:#fee
  style RC fill:#ffe
  style SI fill:#efe
  style SER fill:#eef
```

| Isolation level | Dirty read | Dirty write | Read skew (non-repeatable read) | Lost update | Write skew / phantom |
|---|---|---|---|---|---|
| Read Uncommitted | possible | prevented | possible | possible | possible |
| Read Committed | prevented | prevented | possible | possible | possible |
| Snapshot Isolation | prevented | prevented | prevented | prevented (most impls) | possible |
| Serializable | prevented | prevented | prevented | prevented | prevented |

Only **Serializable** fully rules out write skew and phantoms. Most databases default to Read Committed; "repeatable read" in PostgreSQL is actually snapshot isolation; Oracle's "serializable" is also just snapshot isolation.

---

## ACID Properties — Deep Dive

ACID was coined in 1983 by Härder and Reuter to give fault-tolerance vocabulary a precise meaning. Over time, vendor implementations diverged so far that "ACID compliant" now carries little technical weight — you must read the fine print of each database's isolation and durability semantics.

The practical dividing lines you care about are:

1. **Atomicity (abortability)** — does an aborted transaction leave no trace? Crucial for safe retries.
2. **Isolation level** — what anomalies can you observe? This is almost always the interesting question.
3. **Durability scope** — fsync to one disk, fsync to replicas, or both? A single disk is not durable.

The opposite acronym **BASE** (Basically Available, Soft state, Eventual consistency) is so vague that "not ACID" is essentially its only definition.

---

## Read Committed Isolation

Read Committed is the workhorse default of most relational databases. It provides two guarantees:

- **No dirty reads** — you only see data that has been committed.
- **No dirty writes** — you can only overwrite data that has been committed.

Dirty writes are prevented with row-level **write locks** held until commit. For dirty reads, the naive approach — read locks — kills concurrency, because a long-running writer would block every reader. Instead, databases keep **both the old committed value and the new uncommitted value** for each row, and serve the old value to everyone until the writer commits.

```mermaid
sequenceDiagram
  participant W as Writer
  participant DB as Database
  participant R as Reader
  W->>DB: UPDATE balance = 900 (uncommitted)
  R->>DB: SELECT balance
  DB-->>R: 1000 (old committed value)
  W->>DB: COMMIT
  R->>DB: SELECT balance
  DB-->>R: 900 (new committed value)
```

Read committed still allows **read skew** (non-repeatable reads): a single long transaction can see two different committed values for the same row if another transaction commits in between. Fixing that requires snapshot isolation.

---

## Snapshot Isolation and MVCC

Snapshot isolation gives each transaction a **consistent view of the entire database as of its start time**. Readers never block writers; writers never block readers. It's implemented using **MVCC** (multiversion concurrency control): every row keeps multiple versions tagged with the transaction IDs that created and deleted them.

```mermaid
graph TD
  subgraph accounts[accounts row id=2]
    V1["balance=500<br/>inserted_by=tx_42<br/>deleted_by=tx_99"]
    V2["balance=400<br/>inserted_by=tx_99<br/>deleted_by=null"]
  end
  T[tx_75 snapshot<br/>sees only tx_ids <= 75<br/>and not in the in-progress set]
  T -.visible.-> V1
  T -.invisible.-> V2
```

A version is visible to a transaction `T` if its `inserted_by` tx is committed before `T` started, and its `deleted_by` tx is either null or not yet committed as of `T`'s start. Garbage collection reclaims versions once no active transaction could see them.

Naming is a mess: PostgreSQL calls this "Repeatable Read," Oracle calls it "Serializable," and the SQL standard's "Repeatable Read" is weaker still. Treat the labels with suspicion and read the docs.

---

## Preventing Lost Updates

A **lost update** happens when two transactions do a read-modify-write cycle on the same object and one clobbers the other. Example: two clients increment the same counter concurrently, and one increment vanishes.

| Technique | How it works | Pros | Cons |
|---|---|---|---|
| **Atomic database operation** | `UPDATE … SET x = x + 1 WHERE id = …` executed server-side with an exclusive lock on the row. | Simple, built in, fast. | Requires expressing the update as a single atomic primitive; ORMs love to generate read-modify-write code instead. |
| **Explicit lock** | `SELECT … FOR UPDATE` — app locks the row, then modifies, then commits. | Works for complex logic the DB can't express atomically. | Easy to forget. Risk of deadlock. Serialization bottleneck. |
| **Automatic lost-update detection** | DB tracks reads in a transaction and aborts on conflict at commit (PostgreSQL repeatable read, Oracle serializable, Firebase). | No developer discipline required. | App must handle aborts with retry. Not offered by all engines. |
| **Compare-and-set (CAS)** | `UPDATE … SET value = new WHERE id = X AND value = old`. | Works in systems without transactions. Optimistic — no locks. | Needs a correct "has not changed" predicate; stale reads from replicas can fool CAS. |

In **leaderless or multi-leader** replication, locks and CAS are insufficient because each replica accepts writes independently. You must either resolve conflicts explicitly (e.g. merge siblings in Riak) or use last-write-wins and accept lost updates.

---

## Write Skew and Phantoms

**Write skew** is the subtle cousin of lost updates. Two transactions read **overlapping but not identical** data, each makes a decision based on what they read, and each writes to a **different** object. Individually each is legitimate; together they violate an invariant.

Classic example — doctors on call:

```mermaid
sequenceDiagram
  participant A as Alice (tx1)
  participant DB as Database<br/>invariant: >= 1 on_call doctor
  participant B as Bob (tx2)
  A->>DB: SELECT count(*) on_call = true -> 2
  B->>DB: SELECT count(*) on_call = true -> 2
  A->>DB: UPDATE alice SET on_call = false
  B->>DB: UPDATE bob SET on_call = false
  A->>DB: COMMIT
  B->>DB: COMMIT
  Note over DB: Now 0 doctors on call — invariant broken
```

**Phantoms** are the root cause: a write in one transaction changes the result of a search that another transaction already performed. Snapshot isolation doesn't help — both saw the same consistent snapshot.

Other write-skew flavors:

- Meeting room booking (two rooms reserved at the same slot)
- Multiplayer game moves (two players claim the same square)
- Unique usernames (two signups for the same username)
- Double-spending (two payments against the same balance)

Fixes: true serializable isolation, **materializing conflicts** (insert a dummy row the writers can lock on), or predicate/index-range locks.

---

## Serializability — Serial Execution, 2PL, SSI

Serializable isolation guarantees the outcome is equivalent to **some** serial order of the transactions, regardless of what actually happened concurrently. Three very different implementations:

| Approach | Idea | Performance model | Best when |
|---|---|---|---|
| **Serial execution** | Run one transaction at a time on a single thread; use stored procedures to avoid network round-trips inside a tx. (VoltDB, single-threaded Redis.) | Throughput capped by one CPU core. Zero lock overhead. | Dataset fits in RAM; transactions are short; you can shard across cores. |
| **Two-phase locking (2PL)** | Acquire all locks (shared or exclusive) during the transaction, release them only at commit. Predicate/index-range locks prevent phantoms. | Good throughput when contention is low. Deadlocks, latency spikes from lock waits. | Traditional RDBMS workloads; MySQL's serializable level. |
| **Serializable snapshot isolation (SSI)** | Run on MVCC (no locks), track each tx's reads and writes, abort at commit time if a serializable schedule is impossible. (PostgreSQL, CockroachDB, FoundationDB.) | Optimistic — wins big when conflicts are rare, hurts under high contention (many aborts). | Mostly-read workloads; modern NewSQL deployments. |

```mermaid
graph TD
  ser[Serializable] --> se[Serial Execution<br/>single thread]
  ser --> tpl[Two-Phase Locking<br/>pessimistic, blocks]
  ser --> ssi[Serializable Snapshot Isolation<br/>optimistic, aborts on conflict]
```

---

## Two-Phase Commit and Distributed Transactions

When a transaction touches multiple nodes, **atomic commit** across them all requires a dedicated protocol. The canonical one is **2PC**:

```mermaid
sequenceDiagram
  participant C as Coordinator
  participant P1 as Participant 1
  participant P2 as Participant 2
  C->>P1: PREPARE
  C->>P2: PREPARE
  P1-->>C: yes (promise to commit)
  P2-->>C: yes (promise to commit)
  Note over C: commit point:<br/>write decision to coordinator log
  C->>P1: COMMIT
  C->>P2: COMMIT
  P1-->>C: ack
  P2-->>C: ack
```

Once a participant votes **yes**, it has **promised** to be able to commit — it cannot change its mind, even if it later crashes. If the coordinator crashes after participants have voted but before broadcasting the decision, participants are **stuck** — they hold locks and can't unilaterally commit or abort. This blocking behavior is 2PC's biggest weakness.

Deployment flavors:

- **Database-internal 2PC** (Spanner, CockroachDB): all participants run the same DB, and the coordinator is replicated via Raft/Paxos so it doesn't block. This is viable and widely deployed.
- **Heterogeneous XA** transactions across different systems (e.g. DB + message broker): frequently problematic — XA coordinators don't support SSI, heuristics lose atomicity, operational pain is real.
- **Exactly-once message processing** is often better achieved with **idempotent consumers** plus a dedup table than with distributed transactions.

---

## A Small Python Simulation: Lost Update

Below is a ~25-line stdlib-only simulation of a classic lost update. Each worker does a naive read-modify-write on a shared counter without any locking. A barrier forces all threads to start simultaneously; a tiny sleep between read and write forces the interleaving.

In [ ]:
import threading
import time

counter = 0
expected_increments = 1000
num_threads = 10
increments_per_thread = expected_increments // num_threads

barrier = threading.Barrier(num_threads)

def naive_read_modify_write():
    global counter
    barrier.wait()  # force all threads to start together
    for _ in range(increments_per_thread):
        value = counter            # READ
        time.sleep(0)              # yield to another thread
        counter = value + 1        # WRITE

threads = [threading.Thread(target=naive_read_modify_write) for _ in range(num_threads)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Expected counter: {expected_increments}")
print(f"Actual counter:   {counter}")
print(f"Lost updates:     {expected_increments - counter}")

Despite the GIL, the read and write are separate bytecode ops. The sleep and barrier reliably expose the race. In a real database, this is what an atomic `UPDATE x = x + 1`, a `SELECT FOR UPDATE`, or a CAS would prevent.

---

## Serial-Execution Throughput — A Toy Calculation

VoltDB-style serial execution caps total throughput at one core's worth of transactions. A quick back-of-the-envelope:

In [ ]:
# If a single-threaded executor can process 1000 short tx/second...
tps = 1000
tx_per_day = tps * 86400
print(f"Transactions per day: {tx_per_day:,}")

# How long a 10 ms transaction takes the whole system:
tx_latency_ms = 10
max_tps_single_thread = 1000 / tx_latency_ms * 1000  # per second
print(f"Theoretical max TPS at {tx_latency_ms}ms per tx: {max_tps_single_thread:,.0f}")

# Horizontal scaling via sharding: N cores on independent partitions
cores = 16
print(f"With {cores} disjoint shards: ~{max_tps_single_thread * cores:,.0f} TPS total")

Serial execution trades concurrency complexity for raw single-thread speed — acceptable only because modern machines are fast and stored procedures keep tx latency low.

---

## Trade-offs and Alternatives: Three Paths to Serializability

| Aspect | Serial Execution | Two-Phase Locking (2PL) | Serializable Snapshot Isolation (SSI) |
|---|---|---|---|
| **Concurrency model** | None — one tx at a time | Pessimistic locking | Optimistic, snapshot-based |
| **Readers block writers?** | N/A | Yes (shared/exclusive locks) | No |
| **Writers block readers?** | N/A | Yes | No |
| **Behavior under contention** | Queues up — latency grows | Lock waits, deadlocks, thrashing | High abort rate — retry storm |
| **Scaling story** | Shard across cores/nodes (cross-shard tx is expensive) | Vertical scaling; distributed 2PL is painful | Distributes naturally (per-row MVCC) |
| **Latency sensitivity** | Needs stored procedures so tx latency is sub-ms | Bad for long transactions (lock hold time) | Works for long read-mostly tx |
| **Deployed in** | VoltDB, single-threaded Redis | MySQL InnoDB serializable, SQL Server | PostgreSQL, CockroachDB, FoundationDB |
| **Best for** | OLTP with short, well-partitioned tx | Moderate concurrency, known workloads | Read-heavy, low-contention workloads |

---

## Key Takeaways

1. **Transactions exist to simplify the programming model**, not because the database needs them — every guarantee you drop, the app has to implement itself.
2. **Atomicity is really "abortability"**: its value is making retry safe, not preventing concurrent interference.
3. **"ACID-compliant" is marketing** — always ask which isolation level is the default and what anomalies it actually prevents.
4. **Read committed is the common default**, but it still permits lost updates, read skew, and write skew. Snapshot isolation (MVCC) plugs most of those gaps except write skew / phantoms.
5. **Lost updates have four canonical fixes**: atomic ops, explicit locks, automatic detection, and compare-and-set. Pick one; don't rely on the ORM.
6. **Write skew is the sneakiest anomaly** because both transactions see a consistent snapshot and each write is "legal." Only serializable isolation (or materialized conflicts) kills it.
7. **Serializable can be achieved three ways** — serial execution, 2PL, and SSI — each with a distinct performance profile. SSI is the modern default in new distributed SQL systems.
8. **2PC works but blocks on coordinator failure**; heterogeneous XA is often more trouble than it's worth, while database-internal distributed transactions (Spanner, CockroachDB) have become a practical foundation for scalable ACID.

---

## See Also

- [[01-acid-properties]]
- [[02-read-committed-isolation]]
- [[03-snapshot-isolation-mvcc]]
- [[04-preventing-lost-updates]]
- [[05-write-skew-and-phantoms]]
- [[06-serializability-techniques]]
- [[07-two-phase-commit-distributed]]